# JavaScript — Modern syntax

## LESSON 32 — Optional chaining and nullish coalescing

Real data is often incomplete: a user with no address, a product with no discount, a server response with a field left out. This lesson covers a handful of short operators for one situation — *"this value might not be there"* — so you can handle it without crashes and without a pile of `if` statements. Once you start fetching data from a server in LESSON 48, you will use them all the time.

### First: the two "empty" values

You met these in LESSON 5:

- `undefined` → there is no value. You get it, for example, when you read a key the object does not have.
- `null` → a value someone set to "empty" on purpose.

Together they are called **nullish** values. The word comes back later in this lesson, so keep it in mind: **nullish means `null` or `undefined` — nothing else.**

### `?.` — stop instead of crashing

Reading a key that does not exist is harmless: you just get `undefined`. The trouble is the **next** step. Asking `undefined` for one of *its* keys throws an error, and your program stops right there.

```js
const user = { name: "Mia" };

user.address;            // undefined   -> fine, the key just isn't there
user.address.city;       // TypeError   -> crash: undefined has no keys at all
user.address?.city;      // undefined   -> no crash
```

Read `?.` as a question: *"does the thing on my left exist?"*

- **Yes** → JavaScript carries on, exactly as if you had written a plain `.`.
- **No** (it is `null` or `undefined`) → JavaScript stops there, and the whole expression gives `undefined`. No error.

It also works when you call a method or read an index. The `?.` goes right before the `(` or the `[` — it looks strange at first, but that is the only way to write it:

```js
user.greet?.();          // calls greet if it exists, otherwise gives undefined
list?.[0];               // reads index 0 if list exists, otherwise gives undefined
```

### Truthy and falsy

Before `??` makes sense, you need to know how `||` really behaves — and that comes down to one idea.

Whenever JavaScript needs a yes-or-no answer (in an `if`, or around `||` and `&&`), it can treat **any** value as true or false, not only booleans. Just a few values count as false. They are called **falsy**:

| falsy value | what it is |
|---|---|
| `false` | the boolean |
| `0` | zero (and also `-0` and `NaN`) |
| `""` | an empty string |
| `null` | empty on purpose |
| `undefined` | no value |

Every other value is **truthy** — `1`, `"hello"`, and even `"0"`, `[]` and `{}`. An empty array still counts as true.

To see which group a value belongs to, pass it to `Boolean`:

```js
Boolean(0);        // false
Boolean("hi");     // true
Boolean([]);       // true
```

### `||` — take the first truthy value

In LESSON 7 you used `||` between booleans: `true || false`. It works with any value, though, and what it gives back is not always `true` or `false`:

`a || b` means *"if `a` is truthy, give me `a`; otherwise give me `b`."*

```js
"Mia" || "Anonymous";   // "Mia"         -> "Mia" is truthy, so it is kept
"" || "Anonymous";      // "Anonymous"   -> "" is falsy, so the backup is used
```

That makes `||` a quick way to supply a **default**: a backup value to use when the real one is missing. But there is a catch. `||` replaces **every** falsy value, including ones that are perfectly valid:

```js
const count = 0;

count || 10;      // 10   <- wrong: 0 is a real count, but it is falsy
```

### `??` — a default only when the value is really missing

`??` does the same job, but it only uses the backup when the left side is **nullish** (`null` or `undefined`). `0`, `""` and `false` are real values, so `??` keeps them.

`a ?? b` means *"if `a` is `null` or `undefined`, give me `b`; otherwise give me `a`."*

```js
const count = 0;

count || 10;      // 10   <- 0 is falsy, so || throws it away
count ?? 10;      // 0    <- 0 is not nullish, so ?? keeps it
```

Side by side:

| left side | `left \|\| "default"` | `left ?? "default"` |
|---|---|---|
| `"Mia"` | `"Mia"` | `"Mia"` |
| `0` | **`"default"`** | `0` |
| `""` | **`"default"`** | `""` |
| `false` | **`"default"`** | `false` |
| `null` | `"default"` | `"default"` |
| `undefined` | `"default"` | `"default"` |

The two only disagree on `0`, `""` and `false` — and that is exactly where the bugs hide.

`?.` and `??` are often used together: `?.` reads safely, `??` fills the gap.

```js
user.address?.city ?? "unknown";   // "unknown"
```

### Logical assignment

These three are shortcuts for *"change this value, but only in a certain case"*. Each one pairs an operator you have just seen with `=`.

```js
settings.theme ??= "light";    // if theme is null or undefined, set it to "light"
name ||= "Anonymous";          // if name is falsy, set it to "Anonymous"
isReady &&= isOnline;          // if isReady is truthy, replace it with isOnline
```

Written out the long way:

| shortcut | does the same as |
|---|---|
| `a ??= b` | `if (a === null \|\| a === undefined) a = b;` |
| `a \|\|= b` | `if (!a) a = b;` |
| `a &&= b` | `if (a) a = b;` |

`??=` is the one you will use most, usually to fill in a setting nobody has chosen yet.

### Key notes

- **`?.` only checks what is directly on its left.** In `a?.b.c`, it protects the step from `a` to `b`, but not the step from `b` to `c`. If `a.b` is missing, `.c` still crashes. Guard every step that might be missing: `a?.b?.c`.
- **For defaults, use `??` unless you really mean "any falsy value".** The bug `||` causes stays hidden until a real `0` or an empty string turns up.
- **`?.` is for reading, not writing.** `user.address?.city = "Rome"` is a `SyntaxError`.
- **`?.` is not error handling.** It hides a missing value; it does not tell you *why* the value is missing. Use it where "missing" is normal and expected — not to silence a bug.
- These operators look like noise at first. Before long, they become the shortest honest way to say "this might not exist".

### Example

In [ ]:
const exampleUser = {
  name: "Mia",
  settings: { theme: "dark" },
};

console.log(exampleUser.address?.city);
console.log(exampleUser.settings?.theme);
console.log(exampleUser.greet?.());
console.log(exampleUser.address?.city ?? "unknown");

console.log(Boolean(0));
console.log(Boolean("0"));

const exampleCount = 0;
console.log(exampleCount || 10);
console.log(exampleCount ?? 10);

const exampleConfig = { theme: null };
exampleConfig.theme ??= "light";
console.log(exampleConfig.theme);

### Exercise

Given:

```js
const order = {
  id: 7,
  customer: { name: "Alex" },
  discount: 0,
};
```

1. Print the customer's name.
2. Print `order.shipping.city` without crashing. The object has no `shipping` key.
3. Print the discount with `10` as the default. The real `0` must survive.
4. Do the same with `||` instead, and compare the two results.
5. Add a `note` key with the value `"none"`, but only if `note` is missing.

_Don't open `solutions.ipynb` until you've actually tried._

In [ ]:
// Your code here

### Mini challenge

Given:

```js
const responses = [
  { user: { profile: { city: "Rome" } } },
  { user: { profile: {} } },
  { user: {} },
  {},
];
```

Loop over them and print the city of each, falling back to `"unknown"` when any step of the path is missing. All four must print without a single crash.

In [ ]:
// Your code here